In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, 
    roc_auc_score, confusion_matrix
)
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE, SMOTE, SMOTENC,ADASYN
import numpy as np
import pandas as pd
import time
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, 
    roc_auc_score, confusion_matrix
)
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE

# Data

In [2]:
df_train = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_train.csv")

df_test = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_test.csv")

In [3]:
columns_remove = [
    'VitaminD',
    'YearStart',
]

In [4]:
df_train['ALT_bin'] = pd.cut(df_train['ALT'], bins=[0, 40, 80, np.inf], labels=[0,1,2])
df_train['AST_bin'] = pd.cut(df_train['AST'], bins=[0, 40, 80, np.inf], labels=[0,1,2])


In [5]:
df_test['ALT_bin'] = pd.cut(df_test['ALT'], bins=[0, 40, 80, np.inf], labels=[0,1,2])
df_test['AST_bin'] = pd.cut(df_test['AST'], bins=[0, 40, 80, np.inf], labels=[0,1,2])


In [5]:
df_train['AST_ALT_ratio'] = df_train['AST'] / (df_train['ALT'] + 1e-6)
df_test['AST_ALT_ratio'] = df_test['AST'] / (df_test['ALT'] + 1e-6)

In [6]:
df_train = df_train[df_train['milk_consumption']<=3]
df_test = df_test[df_test['milk_consumption']<=3]

In [7]:
df_train.drop(columns=columns_remove, inplace=True)
df_test.drop(columns=columns_remove, inplace=True)

In [9]:
category_columns = [
    'Gender','Race' ,'SmokeFam','label','ALT_bin','AST_bin','milk_consumption'
]

In [10]:
unuseful_features = ['SmokeFam','WaistCircumference','AST','ALT','AlkalinePhosphotase','UricAcid','LDLCholesterol','Hematocrit','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume']

In [11]:
df_train.drop(columns=unuseful_features,inplace=True)
df_test = df_test[df_train.columns]

In [12]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI', 'FastingGlucose',
       'Triglycerides', 'Creatinine', 'HDLCholesterol', 'Hemoglobin',
       'MeanCellVolumn', 'RedCellDistributionWidth', 'Hba1c',
       'milk_consumption', 'label', 'ALT_bin', 'AST_bin'],
      dtype='object')

# Training and evaluation

## stacking setup

In [14]:
#model definition
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}
gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}

rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
base_learners = [
    ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    #('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
    ('RandomForest', rf_setups['Regularized']),
    # ('dt', dt_setups['Shallow']),  
    # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    # ('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
    ('nb', GaussianNB(var_smoothing= 1e-10)),
]


# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.6,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

## All classifier setup

In [15]:

classifiers = {
    'LightGBM': LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0),
    'GradiantBoosting': GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42),
    'Naive Bayes': GaussianNB(var_smoothing=1e-10),
    'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42),
    'Stacking':stacking_clf
}

## Training and evaluation

In [ ]:
# ====== 0) Train/test split ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race','milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col !='label']

# ====== 1) Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)
#svmsmote = ADASYN(random_state=42, sampling_strategy="auto", n_neighbors=20)
#SMOTE(random_state=42, sampling_strategy="auto", k_neighbors=20)
# ====== 4) Wrapper for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)


# ====== 5) Train & evaluate ======
rows = []

for name, clf in classifiers.items():
    print(f"\n🚀 Training {name} with SVMSMOTE...")
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', svmsmote),
        ('classifier', clf)
    ])
    wrapped_model = ImblearnWrapper(pipeline)

    try:
        start = time.time()
        wrapped_model.fit(X_train_raw, y_train)
        train_time = time.time() - start

        y_pred = wrapped_model.predict(X_test_raw)
        y_proba = wrapped_model.predict_proba(X_test_raw)

        # ===== Global metrics =====
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # AUC
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

        # ===== Per-class metrics =====
        cm = confusion_matrix(y_test, y_pred, labels=[0,1])
        classes = [0,1]
        for i, cls in enumerate(classes):
            TP = cm[i,i]
            FN = cm[i,:].sum() - TP
            FP = cm[:,i].sum() - TP
            TN = cm.sum() - (TP+FN+FP)

            PPV = TP/(TP+FP) if (TP+FP)>0 else 0  # Precision
            NPV = TN/(TN+FN) if (TN+FN)>0 else 0
            SEN = TP/(TP+FN) if (TP+FN)>0 else 0  # Recall
            SPE = TN/(TN+FP) if (TN+FP)>0 else 0

            # Add row with per-class metrics
            rows.append({
                "Model": name,
                "Label": cls,
                "Training time": round(train_time,4) if cls==0 else None,
                "ACC": round(acc,4) if cls==0 else None,
                "F1_macro": round(f1_macro,4) if cls==0 else None,
                "F1_weighted": round(f1_weighted,4) if cls==0 else None,
                "AUC": round(auc,4) if cls==0 else None,
                "PPV": round(PPV,4),
                "NPV": round(NPV,4),
                "SEN": round(SEN,4),
                "SPE": round(SPE,4),
            })

        print(f"✅ {name} - ACC={acc:.4f}, F1_macro={f1_macro:.4f}, AUC={auc:.4f}")

    except Exception as e:
        print(f"❌ Error training {name}: {e}")


# ====== 6) Save results ======
results_df = pd.DataFrame(rows)

print("\n📊 FINAL TABLE (Excel format-ready):")
print(results_df)

results_df.to_csv("deletenull_keep_alt.csv", index=False)
print("\n✅ Exported to expected_output_table.csv")



🚀 Training LightGBM with SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4871
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 22
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ XGBoost - ACC=0.7095, F1_macro=0.6707, AUC=0.7425

🚀 Training GradiantBoosting with SVMSMOTE...
✅ GradiantBoosting - ACC=0.6916, F1_macro=0.6614, AUC=0.7357

🚀 Training RandomForest with SVMSMOTE...
✅ RandomForest - ACC=0.7184, F1_macro=0.6545, AUC=0.7322

🚀 Training GradientBoosting with SVMSMOTE...
✅ GradientBoosting - ACC=0.6942, F1_macro=0.6641, AUC=0.7381

🚀 Training Naive Bayes with SVMSMOTE...
✅ Naive Bayes - ACC=0.6333, F1_macro=0.6161, AUC=0.6933

🚀 Training SVM with SVMSMOTE...
✅ SVM - ACC=0.6728, F1_macro=0.6482, AUC=0.7140

🚀 Training Stacking with SVMSMOTE...
✅ Stacking - ACC=0.7080, F1_macro=0.6596, AUC=0.7255

📊 FINAL TABLE (Excel format-ready):
               Model  Label  Training time     ACC  F1_macro  F1_weighted  \
0           LightGBM      0         6.0859  0.7069    0.6630       0.7091   
1           LightGBM      1            NaN     NaN       NaN          NaN   
2            XGBoost      0         4.1657  0.7095    0.6707       0.7136   
3            XGBoost 

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
